# Задание 1


In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.postgres.operators.postgres import PostgresOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
from airflow.utils.dates import days_ago
from datetime import datetime, timedelta
import csv
import os

with DAG(
    dag_id='create_initial_tables',
    schedule_interval=timedelta(days=1), # Запускать раз в день,
    start_date=days_ago(1), 
    catchup=False,
    tags=['example', 'file_io', 'postgres'],
    default_args={
        'owner': 'airflow',
        'depends_on_past': False,
        'email_on_failure': False,
        'email_on_retry': False,
        'retries': 1,
    }
) as dag:
    # 1. Задача для создания таблицы в PostgreSQL
    create_customer_table_task = PostgresOperator(
        task_id='create_customer_table_if_not_exists',
        postgres_conn_id='postgres_default',      # подключение через AirFlow UI
        
        sql="""
            drop table if exists customer;
            create table if not exists customer (
                customer_id int4 NULL,
                first_name varchar(50) NULL,
                last_name varchar(50) NULL,
                gender varchar(50) NULL,
                "DOB" varchar(50) NULL,
                job_title varchar(50) NULL,
                job_industry_category varchar(50) NULL,
                wealth_segment varchar(50) NULL,
                deceased_indicator varchar(50) NULL,
                owns_car varchar(50) NULL,
                address varchar(50) NULL,
                postcode int4 NULL,
                state varchar(50) NULL,
                country varchar(50) NULL,
                property_valuation int4 NULL
            );
        """,
    )

    create_product_table_task = PostgresOperator(
        task_id='create_product_table_if_not_exists',
        postgres_conn_id='postgres_default',      # подключение через AirFlow UI
        
        sql="""
            drop table if exists product;
            create table if not exists product (
                product_id int4 NULL,
                brand varchar(50) NULL,
                product_line varchar(50) NULL,
                product_class varchar(50) NULL,
                product_size varchar(50) NULL,
                list_price float4 NULL,
                standard_cost float4 NULL
            );
        """,
    )

    create_orders_table_task = PostgresOperator(
        task_id='create_orders_table_if_not_exists',
        postgres_conn_id='postgres_default',      # подключение через AirFlow UI
        
        sql="""
            drop table if exists orders;
            create table if not exists orders (
                order_id int4 NULL,
                customer_id int4 NULL,
                order_date varchar(50) NULL,
                online_order bool NULL,
                order_status varchar(50) NULL
            );
        """,
    )

    create_order_items_table_task = PostgresOperator(
        task_id='create_order_items_table_if_not_exists',
        postgres_conn_id='postgres_default',      # подключение через AirFlow UI
        
        sql="""
            drop table if exists order_items;
            create table if not exists order_items (
              	order_item_id int4 NULL,
                order_id int4 NULL,
                product_id int4 NULL,
                quantity float4 NULL,
                item_list_price_at_sale float4 NULL,
                item_standard_cost_at_sale float4 NULL
            );
        """,
    )

    load_customer_data = PostgresOperator(
        task_id='load_customer_data',
        postgres_conn_id='postgres_default',
        sql="""
            COPY customer FROM '/tmp/data_files/customer.csv'
            WITH (FORMAT CSV, HEADER true, DELIMITER ';');
        """
    )
    
    load_product_data = PostgresOperator(
        task_id='load_product_data',
        postgres_conn_id='postgres_default',
        sql="""
            COPY product FROM '/tmp/data_files/product.csv'
            WITH (FORMAT CSV, HEADER true, DELIMITER ',');
        """
    )

    load_orders_data = PostgresOperator(
        task_id='load_orders_data',
        postgres_conn_id='postgres_default',
        sql="""
            COPY order FROM '/tmp/data_files/orders.csv'
            WITH (FORMAT CSV, HEADER true, DELIMITER ',');
        """
    )

    load_order_items_data = PostgresOperator(
        task_id='load_order_items_data',
        postgres_conn_id='postgres_default',
        sql="""
            COPY order_items FROM '/tmp/data_files/order_items.csv'
            WITH (FORMAT CSV, HEADER true, DELIMITER ',');
        """
    )
    
    finish_task = BashOperator(
        task_id='end_message',
        bash_command='echo "Success"',
    )
    
    # Определение зависимостей
    create_customer_table_task >> create_product_table_task >> create_orders_table_task >> create_order_items_table_task >> [load_customer_data >> load_product_data >> load_orders_data >> load_order_items_data] >> finish_task
   



# Задание 2

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.postgres.operators.postgres import PostgresOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
from airflow.utils.dates import days_ago
from datetime import datetime, timedelta
import csv
import os


def get_low_data_from_db_write_to_file_hook(**context):
    """Получить данные и записать в файл"""
    hook = PostgresHook(postgres_conn_id='postgres_default')
    data = hook.get_records("""
            WITH total_sales AS (
                    SELECT 
                        o.customer_id,
                        SUM(oi.quantity * oi.item_list_price_at_sale) AS total_amount
                    FROM 
                        orders o
                    JOIN 
                        order_items oi ON o.order_id = oi.order_id
                    GROUP BY 
                        o.customer_id
                )
                SELECT 
                    first_name, last_name, COALESCE(total_amount, 0) AS total_amount
                FROM 
                    customer c
                LEFT JOIN 
                    total_sales ts ON c.customer_id = ts.customer_id
                ORDER BY 
                    total_amount ASC
                LIMIT 3;
            """)
        write_to_file(data,'low.json')
        

def get_top_data_from_db_write_to_file_hook(**context):
    """Получить данные и записать в файл"""
    hook = PostgresHook(postgres_conn_id='postgres_default')
    data = hook.get_records("""
            WITH total_sales AS (
                    SELECT 
                        o.customer_id,
                        SUM(oi.quantity * oi.item_list_price_at_sale) AS total_amount
                    FROM 
                        orders o
                    JOIN 
                        order_items oi ON o.order_id = oi.order_id
                    GROUP BY 
                        o.customer_id
                )
                SELECT 
                    first_name, last_name, COALESCE(total_amount, 0) AS total_amount
                FROM 
                    customer c
                LEFT JOIN 
                    total_sales ts ON c.customer_id = ts.customer_id
                ORDER BY 
                    total_amount DESC
                LIMIT 3;
            """)
    write_to_file(data,'top.json')


def write_to_file(data, file_name):
    #TODO
    pass;


with DAG(
    dag_id='create_initial_tables',
    schedule_interval=timedelta(days=1), # Запускать раз в день,
    start_date=days_ago(1), 
    catchup=False,
    tags=['example', 'file_io', 'postgres'],
    default_args={
        'owner': 'airflow',
        'depends_on_past': False,
        'email_on_failure': False,
        'email_on_retry': False,
        'retries': 1,
    }
) as dag:
   
    get_low = PythonOperator(
        task_id='get_low_data_from_db_write_to_file',
        python_callable=get_low_data_from_db_write_to_file_hook,
        provide_context=True,   
    )

    get_top = PythonOperator(
        task_id='get_top_data_from_db_write_to_file',
        python_callable=get_top_data_from_db_write_to_file_hook,
        provide_context=True,   
    )
    
    [get_low, get_top]
   

